In [1]:
import os
import sys

# Remove old Spark 3.5.9 configuration
os.environ.pop("SPARK_HOME", None)

# Tell PySpark to use the current Python 3.12
os.environ["PYSPARK_PYTHON"] = sys.executable

print("Python:", sys.executable)
print("SPARK_HOME:", os.environ.get("SPARK_HOME"))
print("PYSPARK_PYTHON:", os.environ.get("PYSPARK_PYTHON"))
print("PYSPARK_DRIVER_PYTHON:", os.environ.get("PYSPARK_DRIVER_PYTHON"))

Python: c:\Users\bda\AppData\Local\Programs\Python\Python312\python.exe
SPARK_HOME: None
PYSPARK_PYTHON: c:\Users\bda\AppData\Local\Programs\Python\Python312\python.exe
PYSPARK_DRIVER_PYTHON: jupyter


In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Fire Example") \
    .getOrCreate()

print("Spark version:", spark.version)
print("Spark working")

c:\Users\bda\AppData\Local\Programs\Python\Python312\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Spark version: 4.2.0
Spark working


In [3]:
def create_sparkSession():
    spark = SparkSession.builder.appName("Fire Example").getOrCreate()
    return spark

In [57]:
import pyspark
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.functions import countDistinct, col, desc, row_number
from pyspark.sql.window import Window
filepath = r"D:\ABD-Lab (BDA)\data\sf-fire-calls.csv"

In [58]:
def create_dataframe(spark, filepath):
    df = spark.read.csv(filepath, header=True, inferSchema=True)
    df1 = df.select('CallType', 'CallDate', 'City', 'Zipcode', 'Neighborhood', 'Delay')
    return df1

In [59]:
def clean_dataset(df):
    df1=df.withColumn('Date', to_date(col('CallDate'), 'MM/dd/yyyy')).drop('CallDate')
    df2=df1.withColumn('Year',year(col('Date')))\
           .withColumn('Month',month(col('Date')))\
           .withColumn('Week',weekofyear(col('Date')))
    return df2

In [60]:
spark=create_sparkSession()
df=create_dataframe(spark,filepath)
df=clean_dataset(df)
df.printSchema()

root
 |-- CallType: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Zipcode: integer (nullable = true)
 |-- Neighborhood: string (nullable = true)
 |-- Delay: double (nullable = true)
 |-- Date: date (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- Week: integer (nullable = true)



In [61]:
df.show()

+----------------+----+-------+--------------------+---------+----------+----+-----+----+
|        CallType|City|Zipcode|        Neighborhood|    Delay|      Date|Year|Month|Week|
+----------------+----+-------+--------------------+---------+----------+----+-----+----+
|  Structure Fire|  SF|  94109|     Pacific Heights|     2.95|2002-01-11|2002|    1|   2|
|Medical Incident|  SF|  94124|Bayview Hunters P...|      4.7|2002-01-11|2002|    1|   2|
|Medical Incident|  SF|  94102|          Tenderloin|2.4333334|2002-01-11|2002|    1|   2|
|    Vehicle Fire|  SF|  94110|      Bernal Heights|      1.5|2002-01-11|2002|    1|   2|
|          Alarms|  SF|  94109|    Western Addition|3.4833333|2002-01-11|2002|    1|   2|
|  Structure Fire|  SF|  94105|Financial Distric...|     1.75|2002-01-11|2002|    1|   2|
|          Alarms|  SF|  94112|Oceanview/Merced/...|2.7166667|2002-01-11|2002|    1|   2|
|          Alarms|  SF|  94102|          Tenderloin|1.7833333|2002-01-11|2002|    1|   2|
|Medical I

In [62]:
def mapSeason(data):
    if 2<data<6:
        return 'Spring'
    elif 5<data<9:
        return 'Summer'
    elif 8<data<12:
        return 'Autumn'
    else:
        return 'Winter'
seasonUDF=udf(mapSeason, StringType())
clean_df=df.withColumn('Season', seasonUDF(col('Month')))
clean_df.show()

c:\Users\bda\AppData\Local\Programs\Python\Python312\Lib\site-packages\pyspark\sql\udf.py:116: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
c:\Users\bda\AppData\Local\Programs\Python\Python312\Lib\site-packages\pyspark\sql\udf.py:120: RuntimeWarning: Arrow optimization failed to enable because PyArrow or Pandas is not installed. Falling back to a non-Arrow-optimized UDF.
  warnings.warn(


+----------------+----+-------+--------------------+---------+----------+----+-----+----+------+
|        CallType|City|Zipcode|        Neighborhood|    Delay|      Date|Year|Month|Week|Season|
+----------------+----+-------+--------------------+---------+----------+----+-----+----+------+
|  Structure Fire|  SF|  94109|     Pacific Heights|     2.95|2002-01-11|2002|    1|   2|Winter|
|Medical Incident|  SF|  94124|Bayview Hunters P...|      4.7|2002-01-11|2002|    1|   2|Winter|
|Medical Incident|  SF|  94102|          Tenderloin|2.4333334|2002-01-11|2002|    1|   2|Winter|
|    Vehicle Fire|  SF|  94110|      Bernal Heights|      1.5|2002-01-11|2002|    1|   2|Winter|
|          Alarms|  SF|  94109|    Western Addition|3.4833333|2002-01-11|2002|    1|   2|Winter|
|  Structure Fire|  SF|  94105|Financial Distric...|     1.75|2002-01-11|2002|    1|   2|Winter|
|          Alarms|  SF|  94112|Oceanview/Merced/...|2.7166667|2002-01-11|2002|    1|   2|Winter|
|          Alarms|  SF|  94102

In [63]:
clean_df[clean_df['Season']=='Summer'].show()

+--------------------+----+-------+--------------------+---------+----------+----+-----+----+------+
|            CallType|City|Zipcode|        Neighborhood|    Delay|      Date|Year|Month|Week|Season|
+--------------------+----+-------+--------------------+---------+----------+----+-----+----+------+
|Citizen Assist / ...|  SF|  94124|Bayview Hunters P...|1.8833333|2002-06-01|2002|    6|  22|Summer|
|      Structure Fire|  SF|  94112|       Outer Mission|2.3166666|2002-06-01|2002|    6|  22|Summer|
|              Alarms|  SF|  94105|Financial Distric...|1.9833333|2002-06-01|2002|    6|  22|Summer|
|    Medical Incident|  SF|  94124|Bayview Hunters P...| 9.266666|2002-06-01|2002|    6|  22|Summer|
|    Medical Incident|  SF|  94122|     Sunset/Parkside|1.9166666|2002-06-01|2002|    6|  22|Summer|
|    Medical Incident|  SF|  94115|    Western Addition|3.9833333|2002-06-01|2002|    6|  22|Summer|
|    Medical Incident|  SF|  94109|           Japantown|      2.5|2002-06-01|2002|    6|  2

1 — Get yearly count of fire calls

In [64]:
yearly_count = (clean_df.groupBy("year").count().orderBy("year"))
yearly_count.show()

+----+-----+
|year|count|
+----+-----+
|2000| 5459|
|2001| 7713|
|2002| 8090|
|2003| 8499|
|2004| 8283|
|2005| 8282|
|2006| 8174|
|2007| 8255|
|2008| 8869|
|2009| 8789|
|2010| 9341|
|2011| 9735|
|2012| 9674|
|2013|10020|
|2014|10775|
|2015|11458|
|2016|11609|
|2017|12135|
|2018|10136|
+----+-----+



3 — Which week in 2018 had the most fire calls?

In [65]:
weekly_2018 = (clean_df.filter(col("Year") == 2018).groupBy("Week").count().orderBy(desc("count")))

weekly_2018.limit(1).show()

+----+-----+
|Week|count|
+----+-----+
|  22|  259|
+----+-----+



4 — Monthly count of fire calls based on year

In [66]:
monthly_count = (clean_df.groupBy("Year", "month").count().orderBy("year", "month"))
monthly_count.show(10,  truncate=False)

+----+-----+-----+
|Year|month|count|
+----+-----+-----+
|2000|4    |335  |
|2000|5    |680  |
|2000|6    |585  |
|2000|7    |668  |
|2000|8    |678  |
|2000|9    |655  |
|2000|10   |620  |
|2000|11   |595  |
|2000|12   |643  |
|2001|1    |622  |
+----+-----+-----+
only showing top 10 rows


4 — Monthly count of fire calls based on year

In [67]:
monthly_count_year = (clean_df.filter(col("year") == 2018).groupBy("month", "calltype").count().orderBy("month", desc("count")))

monthly_count_year.show(10, truncate=False)

+-----+-------------------------------+-----+
|month|calltype                       |count|
+-----+-------------------------------+-----+
|1    |Medical Incident               |692  |
|1    |Alarms                         |122  |
|1    |Structure Fire                 |91   |
|1    |Traffic Collision              |42   |
|1    |Citizen Assist / Service Call  |15   |
|1    |Outside Fire                   |14   |
|1    |Gas Leak (Natural and LP Gases)|5    |
|1    |Water Rescue                   |4    |
|1    |Vehicle Fire                   |4    |
|1    |Other                          |3    |
+-----+-------------------------------+-----+
only showing top 10 rows


6 — Top five fire call types for every season

In [68]:
# Q6. Top five call types for every season of selected year

from pyspark.sql.window import Window

selected_year = 2018

season_calltypes = (
    clean_df
    .filter(col("Year") == selected_year)
    .groupBy("Season", "CallType")
    .count()
)

window_spec = (
    Window
    .partitionBy("Season")
    .orderBy(desc("count"))
)

top5_season_calltypes = (
    season_calltypes
    .withColumn("rank", row_number().over(window_spec))
    .filter(col("rank") <= 5)
    .orderBy("Season", "rank")
)

top5_season_calltypes.show(100, truncate=False)

+------+-----------------+-----+----+
|Season|CallType         |count|rank|
+------+-----------------+-----+----+
|Autumn|Medical Incident |1514 |1   |
|Autumn|Alarms           |251  |2   |
|Autumn|Structure Fire   |201  |3   |
|Autumn|Traffic Collision|100  |4   |
|Autumn|Outside Fire     |39   |5   |
|Spring|Medical Incident |2110 |1   |
|Spring|Alarms           |333  |2   |
|Spring|Structure Fire   |261  |3   |
|Spring|Traffic Collision|133  |4   |
|Spring|Other            |36   |5   |
|Summer|Medical Incident |2053 |1   |
|Summer|Alarms           |336  |2   |
|Summer|Structure Fire   |262  |3   |
|Summer|Traffic Collision|121  |4   |
|Summer|Outside Fire     |61   |5   |
|Winter|Medical Incident |1327 |1   |
|Winter|Alarms           |224  |2   |
|Winter|Structure Fire   |182  |3   |
|Winter|Traffic Collision|79   |4   |
|Winter|Outside Fire     |28   |5   |
+------+-----------------+-----+----+



7 — Whether fire type calls are seasonal?

In [69]:
seasonal_calls = (clean_df.filter(col("year")==2018).groupBy("season").count().orderBy(desc("count")))
seasonal_calls.show()

+------+-----+
|season|count|
+------+-----+
|Spring| 3023|
|Summer| 2969|
|Autumn| 2218|
|Winter| 1926|
+------+-----+



In [70]:
##To calculate percentages:
total_calls_2018 = (
    clean_df
    .filter(col("Year") == 2018)
    .count()
)

seasonal_percentage = (
    seasonal_calls
    .withColumn(
        "Percentage",
        round(col("count") / lit(total_calls_2018) * 100, 2)
    )
)

seasonal_percentage.show()

+------+-----+----------+
|season|count|Percentage|
+------+-----+----------+
|Spring| 3023|     29.82|
|Summer| 2969|     29.29|
|Autumn| 2218|     21.88|
|Winter| 1926|      19.0|
+------+-----+----------+



8 — Which months in 2018 had the highest fire calls?

In [71]:
monthly_2018 = (clean_df.filter(col("year")==2018).groupBy("month").count().orderBy(desc("count")))
monthly_2018.show()

+-----+-----+
|month|count|
+-----+-----+
|   10| 1068|
|    5| 1047|
|    3| 1029|
|    8| 1021|
|    1| 1007|
|    6|  974|
|    7|  974|
|    9|  951|
|    4|  947|
|    2|  919|
|   11|  199|
+-----+-----+



In [72]:
year_calltype_count = (clean_df.groupBy("year", "calltype").count())

window_year = (Window.partitionBy("year").orderBy(desc("count")))

major_calltype_each_year = (
    year_calltype_count.withColumn("rank", row_number().over(window_year)).filter(col("rank") == 1).orderBy("Year")
)
major_calltype_each_year.show(100, truncate=False)

+----+----------------+-----+----+
|year|calltype        |count|rank|
+----+----------------+-----+----+
|2000|Medical Incident|3408 |1   |
|2001|Medical Incident|4653 |1   |
|2002|Medical Incident|5046 |1   |
|2003|Medical Incident|5056 |1   |
|2004|Medical Incident|5137 |1   |
|2005|Medical Incident|5084 |1   |
|2006|Medical Incident|5027 |1   |
|2007|Medical Incident|5114 |1   |
|2008|Medical Incident|5692 |1   |
|2009|Medical Incident|5671 |1   |
|2010|Medical Incident|6186 |1   |
|2011|Medical Incident|6413 |1   |
|2012|Medical Incident|6296 |1   |
|2013|Medical Incident|6690 |1   |
|2014|Medical Incident|7176 |1   |
|2015|Medical Incident|7812 |1   |
|2016|Medical Incident|7999 |1   |
|2017|Medical Incident|8330 |1   |
|2018|Medical Incident|7004 |1   |
+----+----------------+-----+----+



10 — Average delay for each call type

In [73]:
avg_delay_calltype = (clean_df.groupBy("calltype").agg(
    round(avg("delay"), 2).alias("Average_delay")
).orderBy(desc("Average_delay")))

avg_delay_calltype.show(100, truncate=False)

+--------------------------------------------+-------------+
|calltype                                    |Average_delay|
+--------------------------------------------+-------------+
|Mutual Aid / Assist Outside Agency          |38.42        |
|Assist Police                               |26.98        |
|Train / Rail Incident                       |16.45        |
|Administrative                              |12.26        |
|HazMat                                      |7.53         |
|Marine Fire                                 |6.93         |
|Confined Space / Structure Collapse         |6.92         |
|Watercraft in Distress                      |6.89         |
|Suspicious Package                          |6.58         |
|High Angle Rescue                           |6.05         |
|Other                                       |5.51         |
|Water Rescue                                |5.51         |
|Fuel Spill                                  |5.49         |
|Citizen Assist / Servic

11 — Call type with maximum average delay

In [74]:
max_avg_delay = (clean_df.groupBy("calltype").agg(
    round(avg("delay"), 2).alias("Average_delay")
).orderBy(desc("Average_delay")).limit(1))

max_avg_delay.show(truncate=False)

+----------------------------------+-------------+
|calltype                          |Average_delay|
+----------------------------------+-------------+
|Mutual Aid / Assist Outside Agency|38.42        |
+----------------------------------+-------------+



12 — Neighborhood generating the most fire calls in 2018

In [75]:
nighborhood_calls_2018 = (clean_df.filter(col("year")==2018).groupBy("Neighborhood").count().orderBy(desc("count")))
nighborhood_calls_2018.show(truncate=False)

+------------------------------+-----+
|Neighborhood                  |count|
+------------------------------+-----+
|Tenderloin                    |1393 |
|South of Market               |1053 |
|Mission                       |913  |
|Financial District/South Beach|772  |
|Bayview Hunters Point         |522  |
|Western Addition              |352  |
|Sunset/Parkside               |346  |
|Nob Hill                      |295  |
|Hayes Valley                  |291  |
|Outer Richmond                |262  |
|Castro/Upper Market           |251  |
|North Beach                   |231  |
|Excelsior                     |212  |
|West of Twin Peaks            |210  |
|Potrero Hill                  |210  |
|Chinatown                     |191  |
|Pacific Heights               |191  |
|Marina                        |191  |
|Mission Bay                   |178  |
|Bernal Heights                |170  |
+------------------------------+-----+
only showing top 20 rows


13 — Neighborhoods with worst response times in 2018

In [76]:
worst_responce_nigh = (
    clean_df
    .filter(
        (col("Year") == 2018) &
        col("Neighborhood").isNotNull()
    )
    .groupBy("Neighborhood")
    .agg(
        round(avg("Delay"), 2).alias("Average_Delay"),
        count("*").alias("Total_Calls")
    )
    .orderBy(desc("Average_Delay"))
)
worst_responce_nigh.show()

+--------------------+-------------+-----------+
|        Neighborhood|Average_Delay|Total_Calls|
+--------------------+-------------+-----------+
|           Chinatown|         6.19|        191|
|            Presidio|         5.83|         69|
|     Treasure Island|         5.45|         72|
|        McLaren Park|         4.74|         14|
|Bayview Hunters P...|         4.62|        522|
|    Presidio Heights|         4.59|         71|
|        Inner Sunset|         4.44|        154|
|      Inner Richmond|         4.36|        129|
|Financial Distric...|         4.34|        772|
|      Haight Ashbury|         4.27|        140|
|            Seacliff|         4.26|         15|
|  West of Twin Peaks|         4.19|        210|
|        Potrero Hill|         4.19|        210|
|     Pacific Heights|         4.18|        191|
|          Tenderloin|          4.1|       1393|
|Oceanview/Merced/...|         3.95|        139|
|           Excelsior|         3.94|        212|
|         North Beac

14 — Call type whose average response delay increases/decreases/no relation over years

In [77]:
yearly_delay = (clean_df.groupBy("year", "calltype").agg(
    avg("delay").alias("Average_Delay")
).orderBy("calltype", "year"))
yearly_delay.show(truncate=False)



+----+------------------+------------------+
|year|calltype          |Average_Delay     |
+----+------------------+------------------+
|2005|Administrative    |31.983334         |
|2006|Administrative    |1.8               |
|2017|Administrative    |3.0               |
|2000|Aircraft Emergency|3.905555533333333 |
|2001|Aircraft Emergency|2.616666675       |
|2002|Aircraft Emergency|4.14666662        |
|2003|Aircraft Emergency|13.166667         |
|2004|Aircraft Emergency|2.5916667         |
|2005|Aircraft Emergency|4.29166675        |
|2006|Aircraft Emergency|3.2111111166666664|
|2007|Aircraft Emergency|3.094444333333333 |
|2009|Aircraft Emergency|3.0083335         |
|2011|Aircraft Emergency|3.5944443333333336|
|2012|Aircraft Emergency|4.65833335        |
|2013|Aircraft Emergency|2.4333334         |
|2014|Aircraft Emergency|7.75              |
|2015|Aircraft Emergency|1.1333333         |
|2000|Alarms            |3.0111468393086813|
|2001|Alarms            |2.6232156389407737|
|2002|Alar

In [78]:
delay_trend = (yearly_delay.groupBy("calltype").agg(
    round(corr("year", "Average_Delay"), 3).alias("Correlation")
))
delay_trend.show(truncate=False)

+--------------------------------------------+-----------+
|calltype                                    |Correlation|
+--------------------------------------------+-----------+
|Elevator / Escalator Rescue                 |0.271      |
|Marine Fire                                 |0.287      |
|Aircraft Emergency                          |-0.161     |
|Confined Space / Structure Collapse         |-0.133     |
|Administrative                              |-0.534     |
|Alarms                                      |0.601      |
|Odor (Strange / Unknown)                    |0.512      |
|Citizen Assist / Service Call               |0.367      |
|HazMat                                      |-0.218     |
|Watercraft in Distress                      |0.391      |
|Explosion                                   |0.389      |
|Oil Spill                                   |0.213      |
|Vehicle Fire                                |-0.093     |
|Suspicious Package                          |0.558     

In [79]:
# Classify the trend

trend_result = (
    delay_trend
    .withColumn(
        "Trend",
        when(col("Correlation") > 0.3, "Increases")
        .when(col("Correlation") < -0.3, "Decreases")
        .otherwise("No Relation")
    )
    .orderBy("Trend", "CallType")
)

trend_result.show(100, truncate=False)

+--------------------------------------------+-----------+-----------+
|calltype                                    |Correlation|Trend      |
+--------------------------------------------+-----------+-----------+
|Administrative                              |-0.534     |Decreases  |
|Assist Police                               |-0.345     |Decreases  |
|Alarms                                      |0.601      |Increases  |
|Citizen Assist / Service Call               |0.367      |Increases  |
|Explosion                                   |0.389      |Increases  |
|Fuel Spill                                  |0.314      |Increases  |
|Industrial Accidents                        |0.364      |Increases  |
|Mutual Aid / Assist Outside Agency          |0.919      |Increases  |
|Odor (Strange / Unknown)                    |0.512      |Increases  |
|Other                                       |0.555      |Increases  |
|Outside Fire                                |0.303      |Increases  |
|Smoke

In [80]:
# Call type with maximum overall average delay

maximum_delay_calltype = (
    clean_df
    .groupBy("CallType")
    .agg(
        round(avg("Delay"), 2).alias("Average_Delay")
    )
    .orderBy(desc("Average_Delay"))
    .limit(1)
)

maximum_delay_calltype.show(truncate=False)

+----------------------------------+-------------+
|CallType                          |Average_Delay|
+----------------------------------+-------------+
|Mutual Aid / Assist Outside Agency|38.42        |
+----------------------------------+-------------+



15 — For each year, which city has more call types?

In [81]:
# Q15. City having the most distinct call types in each year

city_calltypes = (
    clean_df
    .groupBy("Year", "City")
    .agg(
        countDistinct("CallType").alias("Distinct_CallTypes")
    )
)

window_city = (
    Window
    .partitionBy("Year")
    .orderBy(desc("Distinct_CallTypes"))
)

city_with_most_calltypes = (
    city_calltypes
    .withColumn("rank", row_number().over(window_city))
    .filter(col("rank") == 1)
    .orderBy("Year")
)

city_with_most_calltypes.show(100, truncate=False)

+----+-------------+------------------+----+
|Year|City         |Distinct_CallTypes|rank|
+----+-------------+------------------+----+
|2000|SF           |18                |1   |
|2001|SF           |20                |1   |
|2002|SF           |20                |1   |
|2003|SF           |24                |1   |
|2004|SF           |23                |1   |
|2005|SF           |26                |1   |
|2006|SF           |24                |1   |
|2007|SF           |26                |1   |
|2008|SF           |23                |1   |
|2009|SF           |22                |1   |
|2010|SF           |23                |1   |
|2011|SF           |25                |1   |
|2012|SF           |25                |1   |
|2013|SF           |24                |1   |
|2014|San Francisco|21                |1   |
|2015|San Francisco|24                |1   |
|2016|San Francisco|24                |1   |
|2017|San Francisco|26                |1   |
|2018|San Francisco|20                |1   |
+----+----

16 — Every year, count of calltypes for 5 cities with more calls

In [82]:
# Q16 - Step 1
# Find five cities having the most calls

top_5_cities = (
    clean_df
    .groupBy("City")
    .count()
    .orderBy(desc("count"))
    .limit(5)
)

top_5_cities.show()

+-------------+------+
|         City| count|
+-------------+------+
|           SF|120072|
|San Francisco| 51739|
|SAN FRANCISCO|  1676|
|           TI|   486|
|     Presidio|   281|
+-------------+------+



In [83]:
top_cities_list = [
    row["City"]
    for row in top_5_cities.collect()
]

top_cities_list

['SF', 'San Francisco', 'SAN FRANCISCO', 'TI', 'Presidio']

In [84]:
# Q16 - Step 2
# Yearly call count and number of call types for the top 5 cities

yearly_top5_cities = (
    clean_df
    .filter(col("City").isin(top_cities_list))
    .groupBy("Year", "City")
    .agg(
        count("*").alias("Total_Calls"),
        countDistinct("CallType").alias("CallType_Count")
    )
    .orderBy("Year", desc("Total_Calls"))
)

yearly_top5_cities.show(200, truncate=False)

+----+-------------+-----------+--------------+
|Year|City         |Total_Calls|CallType_Count|
+----+-------------+-----------+--------------+
|2000|SF           |5435       |18            |
|2000|TI           |8          |6             |
|2001|SF           |7656       |20            |
|2001|TI           |37         |6             |
|2002|SF           |8044       |20            |
|2002|TI           |18         |5             |
|2003|SF           |8441       |24            |
|2003|TI           |33         |3             |
|2004|SF           |8224       |23            |
|2004|TI           |32         |3             |
|2005|SF           |8228       |26            |
|2005|TI           |31         |7             |
|2006|SF           |8114       |24            |
|2006|TI           |28         |7             |
|2007|SF           |8190       |26            |
|2007|TI           |44         |9             |
|2008|SF           |8811       |23            |
|2008|TI           |36         |8       

17 — Correlation between neighborhood, zip code and number of fire calls

In [85]:
# Q17. Number of calls by Neighborhood and Zipcode

neighborhood_zip_calls = (
    clean_df
    .filter(
        col("Neighborhood").isNotNull() &
        col("Zipcode").isNotNull()
    )
    .groupBy("Neighborhood", "Zipcode")
    .count()
    .withColumnRenamed("count", "Fire_Call_Count")
)

neighborhood_zip_calls.show(50, truncate=False)

+------------------------------+-------+---------------+
|Neighborhood                  |Zipcode|Fire_Call_Count|
+------------------------------+-------+---------------+
|Inner Sunset                  |94122  |2161           |
|Bayview Hunters Point         |94124  |9150           |
|Inner Sunset                  |94114  |20             |
|West of Twin Peaks            |94112  |760            |
|Haight Ashbury                |94114  |21             |
|Glen Park                     |94110  |25             |
|Excelsior                     |94112  |3237           |
|Russian Hill                  |94109  |2261           |
|None                          |94124  |7              |
|Chinatown                     |94133  |1861           |
|Pacific Heights               |94115  |2100           |
|Oceanview/Merced/Ingleside    |94127  |12             |
|Potrero Hill                  |94103  |5              |
|Inner Sunset                  |94117  |224            |
|Golden Gate Park              